In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -qq huggingface_hub

In [3]:
from huggingface_hub import login
login()

In [4]:
from datasets import load_dataset
load_data=load_dataset("karthiksagarn/astro_horoscope",split="train")

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

horoscope.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21959 [00:00<?, ? examples/s]

In [5]:
load_data.column_names

['sign', 'category', 'date', 'horoscope']

In [6]:
!pip install transformers

In [7]:
!nvidia-smi

Mon Apr 20 20:06:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
model_name="Qwen/Qwen3-0.6B"
tokenizer=AutoTokenizer.from_pretrained(model_name)
def tokenize(batch):
    return tokenizer(
        batch["horoscope"],
        truncation=True,
        max_length=1024,
    )
dataset = load_data.map(tokenize, batched=True, remove_columns=load_data.column_names)
dataset = dataset.train_test_split(test_size=0.1)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Map:   0%|          | 0/21959 [00:00<?, ? examples/s]

In [9]:
dataset.column_names

{'train': ['input_ids', 'attention_mask'],
 'test': ['input_ids', 'attention_mask']}

In [10]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False),

In [11]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto")

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [15]:
training_args = TrainingArguments(
    output_dir="nvidia-gemma4-4bit-finetuned",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.568978,2.557086
2,2.417496,2.473003


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1236, training_loss=2.59176772157737, metrics={'train_runtime': 9308.8632, 'train_samples_per_second': 4.246, 'train_steps_per_second': 0.133, 'total_flos': 1.567503881404416e+16, 'train_loss': 2.59176772157737, 'epoch': 2.0})

In [ ]:
model_file=model.push_to_hub("HF_NAME/Qwen3-0.6B-16bit", token = "HF_TOKEN")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [19]:
model_file

CommitInfo(commit_url='https://huggingface.co/Pt-kunal-mishra/Qwen3-0.6B-16bit/commit/6daec54968dfb856d98778618592657c17ac81ec', commit_message='Upload Qwen3ForCausalLM', commit_description='', oid='6daec54968dfb856d98778618592657c17ac81ec', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Pt-kunal-mishra/Qwen3-0.6B-16bit', endpoint='https://huggingface.co', repo_type='model', repo_id='Pt-kunal-mishra/Qwen3-0.6B-16bit'), pr_revision=None, pr_num=None)